<a href="https://www.kaggle.com/code/n07kiran/multiclass-transformed-anerbc-ii-mobilenetv2-ipynb?scriptVersionId=318983606" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!git clone https://github.com/n07kiran/fyp.git
%cd /kaggle/working/fyp
!ls

Cloning into 'fyp'...
remote: Enumerating objects: 1587, done.
remote: Counting objects: 100% (435/435), done.
remote: Compressing objects: 100% (379/379), done.
remote: Total 1587 (delta 96), reused 387 (delta 53), pack-reused 1152 (from 1)
Receiving objects: 100% (1587/1587), 395.57 MiB | 29.38 MiB/s, done.
Resolving deltas: 100% (257/257), done.
Filtering content: 100% (157/157), 3.38 GiB | 27.58 MiB/s, done.
/kaggle/working/fyp
 Code			 __pycache__	   'research papers'   viva_questions
 datasetTransformation	 requirements.txt   transformDataset


In [2]:
!grep -v "tensorflow-metal" requirements.txt > requirements-kaggle.txt
!pip install -r requirements-kaggle.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.6/615.6 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 101.7 MB/s eta 0:00:00
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.19.0
    Uninstalling tensorboard-2.19.0:
      Successfully uninstalled tensorboard-2.19.0
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.19.0
    Uninstalling tensorflow-2.19.0:
      Successfully uninstalled tensorflow-2.19.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
tensorflow-text 2.19.0 requires tensorflow<2.20,>=2.19.0, but you have tensorflow 2.18.1 which is incompatible.
tensorflow-decision-forests 1.12.0 requires tensorflow==2.19.0, but you have tensorflow 2.18.1 which is incompatible.
tf-k

In [3]:
import os
data_root = "/kaggle/input/datasets/n07kiran/transformed-anerbc-dataset"
print(os.listdir(data_root))

['transformed_AneRBC-I', 'transformed_AneRBC-II']


In [4]:
DATASET_PATH = "/kaggle/input/datasets/n07kiran/transformed-anerbc-dataset/transformed_AneRBC-II"

In [5]:
import os
print(os.listdir(DATASET_PATH))

['Macrocytic', 'train_split.csv', 'test_split.csv', 'Microcytic', 'Healthy', 'val_split.csv', 'Normocytic']


Transformed AneRBC-II Multiclass Image + CBC Fusion with MobileNetV2
This notebook trains a two-input fusion model for 4-class anemia classification using the public transformed_AneRBC_dataset splits.

Inputs
Image input: image_path from transformed_AneRBC-II/train_split.csv, transformed_AneRBC-II/val_split.csv, and transformed_AneRBC-II/test_split.csv
CBC input: WBC, RBC, HGB, HCT, MCV, MCH, MCHC, PLT, MPV, RDW_CV
Target: final_class with labels 0=Healthy, 1=Microcytic, 2=Normocytic, 3=Macrocytic
The notebook resolves the dataset locally from transformed_AneRBC_dataset and on Kaggle from n07kiran/transformed-AneRBC-dataset.

Environment Setup
The path setup resolves the read-only transformed dataset separately from the writable artifact directory. On Kaggle, dataset files are read from /kaggle/input and trained outputs are written under /kaggle/working.

In [6]:
from pathlib import Path

KAGGLE_DATASET_ID = "n07kiran/transformed-AneRBC-dataset"
KAGGLE_DATASET_SLUG = "transformed-anerbc-dataset"
KAGGLE_DATASET_SLUGS = (
    KAGGLE_DATASET_SLUG,
    "transformed-AneRBC-dataset",
)
TRANSFORMED_DATASET_DIRNAME = "transformed_AneRBC_dataset"
TRANSFORMED_SUBSET_NAME = "transformed_AneRBC-II"
TRANSFORMED_SUBSET_SLUG = "transformed_aneRBC_ii"


def detect_env() -> str:
    if Path("/kaggle/input").exists():
        return "kaggle"
    try:
        import google.colab  # noqa: F401
        return "colab"
    except ImportError:
        return "local"


ENV = detect_env()
print(f"Detected environment: {ENV}")
print(f"Selected dataset subset: {TRANSFORMED_SUBSET_NAME}")

Detected environment: kaggle
Selected dataset subset: transformed_AneRBC-II


Local TensorFlow Device Setup
On Apple Silicon, tensorflow-metal can crash the native Python process on some training/evaluation graphs. Local notebooks default to CPU for stability. Kaggle GPU behavior is unchanged.

In [7]:
import os
os.environ["TRANSFORMED_ANERBC_DATASET_ROOT"] = "/kaggle/input/datasets/n07kiran/transformed-anerbc-dataset"

In [8]:
import importlib.metadata

if ENV == "local":
    try:
        version = importlib.metadata.version("tensorflow-metal")
        print(
            f"tensorflow-metal {version}: INSTALLED "
            "(local runs default to CPU; set ANERBC_USE_LOCAL_METAL=1 before kernel start to opt in)"
        )
    except importlib.metadata.PackageNotFoundError:
        print("tensorflow-metal not found; local training will use CPU.")
else:
    print(f"ENV={ENV}: GPU visibility is controlled by the hosted runtime.")

ENV=kaggle: GPU visibility is controlled by the hosted runtime.


Imports And Constants
This cell defines the selected transformed dataset subset, backbone-specific settings, split paths, checkpoint lookup, and artifact paths for this notebook.

In [9]:
import tensorflow as tf
import subprocess
import os

print("TensorFlow version:", tf.__version__)
print("Physical GPUs:", tf.config.list_physical_devices('GPU'))

print("\nNVIDIA-SMI:")
result = subprocess.run(["bash", "-lc", "nvidia-smi"], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else result.stderr)

print("\nCUDA visible devices env:", os.environ.get("CUDA_VISIBLE_DEVICES"))

2026-05-13 15:20:13.830489: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778685613.852677      23 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778685613.860035      23 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


TensorFlow version: 2.18.1
Physical GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

NVIDIA-SMI:
Wed May 13 15:20:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             28W /  250W |       3MiB /  16384MiB |      0%      Default |
|                 

In [10]:
import json
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

LOCAL_METAL_GPU_OPT_IN = os.environ.get("ANERBC_USE_LOCAL_METAL", "0") == "1"
if ENV == "local" and not LOCAL_METAL_GPU_OPT_IN:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")

import tensorflow as tf

if ENV == "local" and not LOCAL_METAL_GPU_OPT_IN:
    try:
        tf.config.set_visible_devices([], "GPU")
        print("Local TensorFlow Metal GPU disabled. Set ANERBC_USE_LOCAL_METAL=1 before kernel start to opt in.")
    except RuntimeError as exc:
        print(f"Could not hide local GPU because TensorFlow was already initialized: {exc}")

SEED = 42
BATCH_SIZE = 16
STAGE1_EPOCHS = 30
STAGE2_EPOCHS = 10
PATIENCE = 7

RUN_SMOKE_TEST = True
SMOKE_TRAIN_ROWS = 64
SMOKE_VAL_ROWS = 32

MODEL_SLUG = "mobilenetv2"
MODEL_DISPLAY_NAME = "MobileNetV2"
CHECKPOINT_DATASET_TAG = "anerbc_ii"
BASE_LAYER_NAME = "mobilenetv2_1.00_224"
IMAGE_SIZE = (224, 224)
PREPROCESS_MODE = "tf_minus_one_to_one"
FINE_TUNE_PREFIXES = ()
FINE_TUNE_LAST_N = 20

CLASS_ID_TO_NAME = {
    0: "Healthy",
    1: "Microcytic",
    2: "Normocytic",
    3: "Macrocytic",
}
CLASS_NAMES = [CLASS_ID_TO_NAME[i] for i in range(4)]
NUM_CLASSES = 4

CBC_FEATURES = ["WBC", "RBC", "HGB", "HCT", "MCV", "MCH", "MCHC", "PLT", "MPV", "RDW_CV"]


def parent_candidates(start: Path):
    current = start.resolve()
    yield current
    for parent in current.parents:
        yield parent


def find_repo_root() -> Path:
    override = os.environ.get("ANERBC_REPO_ROOT")
    if override:
        root = Path(override).expanduser().resolve()
        if root.exists():
            return root
        raise FileNotFoundError(f"ANERBC_REPO_ROOT does not exist: {root}")

    for candidate in parent_candidates(Path.cwd()):
        if (candidate / "Code" / "newFusionModel").exists():
            return candidate

    colab_root = Path("/content/drive/MyDrive/Anemia FYP")
    if ENV == "colab" and (colab_root / "Code" / "newFusionModel").exists():
        return colab_root

    return Path.cwd().resolve()


def metadata_matches(path: Path) -> bool:
    metadata_path = path / "dataset-metadata.json"
    if not metadata_path.exists():
        return False
    try:
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    except Exception:
        return False
    return (
        metadata.get("id") == KAGGLE_DATASET_ID
        or metadata.get("title") == TRANSFORMED_DATASET_DIRNAME
    )


def find_transformed_dataset_root() -> Path:
    override = os.environ.get("TRANSFORMED_ANERBC_DATASET_ROOT")
    if override:
        root = Path(override).expanduser().resolve()
        if (root / TRANSFORMED_SUBSET_NAME / "train_split.csv").exists():
            return root
        if root.name == TRANSFORMED_SUBSET_NAME and (root / "train_split.csv").exists():
            return root.parent
        raise FileNotFoundError(
            "TRANSFORMED_ANERBC_DATASET_ROOT must point to transformed_AneRBC_dataset "
            f"or {TRANSFORMED_SUBSET_NAME}: {root}"
        )

    candidates = []
    for base in parent_candidates(Path.cwd()):
        candidates.append(base / TRANSFORMED_DATASET_DIRNAME)
        if base.name == TRANSFORMED_DATASET_DIRNAME:
            candidates.append(base)

    candidates.append(REPO_ROOT / TRANSFORMED_DATASET_DIRNAME)

    if ENV == "colab":
        candidates.append(Path("/content/drive/MyDrive/Anemia FYP") / TRANSFORMED_DATASET_DIRNAME)

    kaggle_input = Path("/kaggle/input")
    if ENV == "kaggle" and kaggle_input.exists():
        for slug in KAGGLE_DATASET_SLUGS:
            candidates.append(kaggle_input / slug)
        candidates.extend(path for path in kaggle_input.iterdir() if path.is_dir())

    seen = set()
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / TRANSFORMED_SUBSET_NAME / "train_split.csv").exists():
            return candidate
        if metadata_matches(candidate) and (candidate / TRANSFORMED_SUBSET_NAME).exists():
            return candidate

    searched = "\n".join(str(path) for path in list(seen)[:20])
    raise FileNotFoundError(
        f"Could not find {TRANSFORMED_DATASET_DIRNAME} with {TRANSFORMED_SUBSET_NAME}. "
        "Attach the public Kaggle dataset n07kiran/transformed-AneRBC-dataset, "
        "keep the local transformed_AneRBC_dataset folder near the repo, or set "
        f"TRANSFORMED_ANERBC_DATASET_ROOT. Checked:\n{searched}"
    )


def unique_existing_roots(*roots: Path) -> list[Path]:
    unique = []
    seen = set()
    for root in roots:
        if root is None:
            continue
        root = root.expanduser().resolve()
        if root.exists() and root not in seen:
            unique.append(root)
            seen.add(root)
    return unique


def resolve_checkpoint_path(names: list[str]) -> Path | None:
    roots = [REPO_ROOT, Path.cwd().resolve()]

    checkpoint_root = os.environ.get("ANERBC_CHECKPOINT_ROOT")
    if checkpoint_root:
        roots.insert(0, Path(checkpoint_root).expanduser())

    kaggle_input = Path("/kaggle/input")
    if ENV == "kaggle" and kaggle_input.exists():
        roots.extend(path for path in kaggle_input.iterdir() if path.is_dir())

    relative_templates = [
        Path("Code") / "multiClassImageClassification" / "artifacts" / "models",
        Path("Code") / "ImageClassification" / "artifacts" / "models",
        Path("multiClassImageClassification") / "artifacts" / "models",
        Path("ImageClassification") / "artifacts" / "models",
        Path("artifacts") / "models",
        Path("."),
    ]

    for root in unique_existing_roots(*roots):
        for name in names:
            for relative_root in relative_templates:
                candidate = root / relative_root / name
                if candidate.exists():
                    return candidate
    return None


def get_output_root() -> Path:
    override = os.environ.get("ANERBC_OUTPUT_ROOT")
    if override:
        return Path(override).expanduser().resolve()
    if ENV == "kaggle":
        return Path("/kaggle/working")
    return REPO_ROOT


REPO_ROOT = find_repo_root()
TRANSFORMED_DATASET_ROOT = find_transformed_dataset_root()
DATASET_ROOT = TRANSFORMED_DATASET_ROOT / TRANSFORMED_SUBSET_NAME
OUTPUT_ROOT = get_output_root()
RUN_NAME = f"multiClass_{TRANSFORMED_SUBSET_SLUG}_{MODEL_SLUG}"

CHECKPOINT_NAMES = [
    f"{MODEL_SLUG}_transfer_frozen_multiclass_{CHECKPOINT_DATASET_TAG}_best.keras",
    f"{MODEL_SLUG}_transfer_frozen_{CHECKPOINT_DATASET_TAG}_best.keras",
]
if CHECKPOINT_DATASET_TAG != "anerbc_i":
    CHECKPOINT_NAMES.append(f"{MODEL_SLUG}_transfer_frozen_multiclass_anerbc_i_best.keras")
CHECKPOINT_PATH = resolve_checkpoint_path(CHECKPOINT_NAMES)

TRAIN_SOURCE_CSV = DATASET_ROOT / "train_split.csv"
VAL_SOURCE_CSV = DATASET_ROOT / "val_split.csv"
TEST_SOURCE_CSV = DATASET_ROOT / "test_split.csv"

ARTIFACTS_DIR = (
    OUTPUT_ROOT
    / "Code"
    / "newFusionModel"
    / "multiClassImageClassification"
    / "artifacts"
    / TRANSFORMED_SUBSET_SLUG
)
MODELS_DIR = ARTIFACTS_DIR / "models"
HISTORY_DIR = ARTIFACTS_DIR / "history"
METRICS_DIR = ARTIFACTS_DIR / "metrics"
PLOTS_DIR = ARTIFACTS_DIR / "plots"

for directory in [ARTIFACTS_DIR, MODELS_DIR, HISTORY_DIR, METRICS_DIR, PLOTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

MODEL_SAVE_PATH = MODELS_DIR / f"{RUN_NAME}_fusion_best.keras"
HISTORY_PATH = HISTORY_DIR / f"{RUN_NAME}_history.csv"
REPORT_PATH = METRICS_DIR / f"{RUN_NAME}_classification_report.txt"
CONFUSION_MATRIX_PATH = PLOTS_DIR / f"{RUN_NAME}_confusion_matrix.png"
SUMMARY_PATH = ARTIFACTS_DIR / f"fusion_results_summary_{TRANSFORMED_SUBSET_SLUG}.csv"

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

VISIBLE_GPUS = tf.config.get_visible_devices("GPU")
if VISIBLE_GPUS:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
else:
    tf.keras.mixed_precision.set_global_policy("float32")

print("TensorFlow version      :", tf.__version__)
print("Visible GPUs            :", VISIBLE_GPUS)
print("Compute dtype policy    :", tf.keras.mixed_precision.global_policy().name)
print("REPO_ROOT               :", REPO_ROOT)
print("TRANSFORMED_DATASET_ROOT:", TRANSFORMED_DATASET_ROOT)
print("DATASET_ROOT            :", DATASET_ROOT)
print("TRAIN_SOURCE_CSV        :", TRAIN_SOURCE_CSV)
print("VAL_SOURCE_CSV          :", VAL_SOURCE_CSV)
print("TEST_SOURCE_CSV         :", TEST_SOURCE_CSV)
print("CHECKPOINT_PATH         :", CHECKPOINT_PATH if CHECKPOINT_PATH else "not found; ImageNet/random base fallback will be used")
print("ARTIFACTS_DIR           :", ARTIFACTS_DIR)

TensorFlow version      : 2.18.1
Visible GPUs            : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Compute dtype policy    : mixed_float16
REPO_ROOT               : /kaggle/working/fyp
TRANSFORMED_DATASET_ROOT: /kaggle/input/datasets/n07kiran/transformed-anerbc-dataset
DATASET_ROOT            : /kaggle/input/datasets/n07kiran/transformed-anerbc-dataset/transformed_AneRBC-II
TRAIN_SOURCE_CSV        : /kaggle/input/datasets/n07kiran/transformed-anerbc-dataset/transformed_AneRBC-II/train_split.csv
VAL_SOURCE_CSV          : /kaggle/input/datasets/n07kiran/transformed-anerbc-dataset/transformed_AneRBC-II/val_split.csv
TEST_SOURCE_CSV         : /kaggle/input/datasets/n07kiran/transformed-anerbc-dataset/transformed_AneRBC-II/test_split.csv
CHECKPOINT_PATH         : /kaggle/working/fyp/Code/ImageClassification/artifacts/models/mobilenetv2_transfer_frozen_anerbc_ii_best.keras
ARTIFACTS_DIR           : /kaggle/working/Code/newFusionModel/multiClassImageClassification/a

Data Loading And Validation
This cell loads the published train/validation/test split CSVs from the selected transformed subset, resolves image paths relative to that subset, validates file resolution, and applies train-median CBC imputation.

In [11]:
def resolve_dataset_path(path_value: str) -> Path:
    path = Path(str(path_value))
    if path.is_absolute():
        return path

    candidates = [
        DATASET_ROOT / path,
        TRANSFORMED_DATASET_ROOT / path,
        REPO_ROOT / path,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]


def load_split(csv_path: Path, split_name: str) -> pd.DataFrame:
    if not csv_path.exists():
        raise FileNotFoundError(f"{split_name} CSV not found: {csv_path}")

    df = pd.read_csv(csv_path)
    required_cols = {"image_path", "final_class", *CBC_FEATURES}
    missing_cols = sorted(required_cols - set(df.columns))
    if missing_cols:
        raise ValueError(f"{split_name} split is missing columns: {missing_cols}")

    for feature in CBC_FEATURES:
        df[feature] = pd.to_numeric(df[feature], errors="coerce")

    df["final_class"] = pd.to_numeric(df["final_class"], errors="raise").astype("int64")
    unexpected = sorted(set(df["final_class"].unique()) - set(range(NUM_CLASSES)))
    if unexpected:
        raise ValueError(f"{split_name} split has unexpected class ids: {unexpected}")
    return df


def compute_inverse_frequency_weights(labels: np.ndarray) -> dict[int, float]:
    labels = labels.astype(np.int64)
    counts = np.bincount(labels, minlength=NUM_CLASSES)
    total = float(labels.size)
    weights = {}
    for class_id, count in enumerate(counts):
        weights[class_id] = total / (NUM_CLASSES * float(count)) if count > 0 else 1.0
    return weights


def apply_train_median_imputation(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.Series]:
    train_medians = train_df[CBC_FEATURES].median(numeric_only=True)
    if train_medians.isna().any():
        missing = train_medians[train_medians.isna()].index.tolist()
        raise ValueError(f"Cannot compute train medians for: {missing}")

    outputs = []
    for split_name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
        split_df = df.copy()
        split_df[CBC_FEATURES] = split_df[CBC_FEATURES].fillna(train_medians)

        remaining_nans = int(split_df[CBC_FEATURES].isna().sum().sum())
        if remaining_nans > 0:
            raise ValueError(f"{split_name} split has {remaining_nans} CBC NaNs after imputation")

        resolved_paths = [resolve_dataset_path(p) for p in split_df["image_path"]]
        missing_images = [str(p) for p in resolved_paths if not p.exists()]
        if missing_images:
            preview = missing_images[:5]
            raise FileNotFoundError(
                f"{split_name} split has missing image files. First 5: {preview}"
            )

        split_df["resolved_image_path"] = [str(p) for p in resolved_paths]
        outputs.append(split_df)

    return outputs[0], outputs[1], outputs[2], train_medians

In [12]:
train_df = load_split(TRAIN_SOURCE_CSV, "train")
val_df = load_split(VAL_SOURCE_CSV, "validation")
test_df = load_split(TEST_SOURCE_CSV, "test")

train_df, val_df, test_df, train_medians = apply_train_median_imputation(train_df, val_df, test_df)

observed_labels = set(train_df["final_class"].unique()) | set(val_df["final_class"].unique()) | set(test_df["final_class"].unique())
expected_labels = set(range(NUM_CLASSES))
if not observed_labels.issubset(expected_labels):
    raise ValueError(f"Unexpected labels found: {sorted(observed_labels - expected_labels)}")

class_weight_map = compute_inverse_frequency_weights(train_df["final_class"].to_numpy(dtype=np.int64))

print(f"Rows: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")
print("Class counts (train):")
print(train_df["final_class"].value_counts().sort_index())
print("Inverse-frequency class weights:", class_weight_map)
print("CBC medians used for imputation:")
print(train_medians)

Rows: train=9048, val=2016, test=1908
Class counts (train):
final_class
0    4200
1    3396
2     732
3     720
Name: count, dtype: int64
Inverse-frequency class weights: {0: 0.5385714285714286, 1: 0.666077738515901, 2: 3.0901639344262297, 3: 3.1416666666666666}
CBC medians used for imputation:
WBC         7.94
RBC         4.70
HGB        10.70
HCT        33.60
MCV        75.70
MCH        24.65
MCHC       32.15
PLT       293.00
MPV        10.90
RDW_CV     14.60
dtype: float64


Dataset Pipeline And Fusion Model
This section builds the two-input Keras model. It reuses only a pretrained image backbone when a compatible local checkpoint is available; otherwise it falls back to a standard Keras application backbone so the notebook can still run in Kaggle with only the public transformed dataset attached.

In [13]:
@tf.keras.utils.register_keras_serializable(package="newFusionModel")
class BackbonePreprocessing(tf.keras.layers.Layer):
    # Backbone preprocessing layer for raw RGB images in the 0..255 range.

    def __init__(self, mode: str, **kwargs):
        super().__init__(**kwargs)
        if mode not in {"vgg16_caffe", "tf_minus_one_to_one", "unit_range"}:
            raise ValueError(f"Unsupported preprocess mode: {mode}")
        self.mode = mode

    def call(self, images):
        images = tf.cast(images, tf.float32)

        if self.mode == "vgg16_caffe":
            bgr = tf.reverse(images, axis=[-1])
            imagenet_bgr_mean = tf.constant([103.939, 116.779, 123.68], dtype=tf.float32)
            return bgr - imagenet_bgr_mean

        if self.mode == "unit_range":
            return images / 255.0

        return (images / 127.5) - 1.0

    def get_config(self):
        config = super().get_config()
        config.update({"mode": self.mode})
        return config


def make_dataset(
    df: pd.DataFrame,
    batch_size: int,
    shuffle: bool,
    seed: int,
    sample_weight_map: dict[int, float] | None = None,
) -> tf.data.Dataset:
    paths = df["resolved_image_path"].astype(str).to_numpy()
    cbc_values = df[CBC_FEATURES].to_numpy(dtype=np.float32)
    labels = df["final_class"].to_numpy(dtype=np.int64)

    if sample_weight_map is None:
        ds = tf.data.Dataset.from_tensor_slices((paths, cbc_values, labels))
    else:
        sample_weights = np.asarray([sample_weight_map[int(lbl)] for lbl in labels], dtype=np.float32)
        ds = tf.data.Dataset.from_tensor_slices((paths, cbc_values, labels, sample_weights))

    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), seed=seed, reshuffle_each_iteration=True)

    def parse_record(path, cbc, label):
        image_bytes = tf.io.read_file(path)
        image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.cast(image, tf.float32)
        image = tf.image.resize(image, IMAGE_SIZE, method="bicubic", antialias=True)
        image = tf.clip_by_value(image, 0.0, 255.0)
        inputs = {
            "image_input": image,
            "cbc_input": cbc,
        }
        return inputs, label

    def parse_weighted_record(path, cbc, label, sample_weight):
        inputs, parsed_label = parse_record(path, cbc, label)
        return inputs, parsed_label, sample_weight

    if sample_weight_map is None:
        ds = ds.map(parse_record, num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.map(parse_weighted_record, num_parallel_calls=tf.data.AUTOTUNE)

    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


def build_application_image_base() -> tf.keras.Model:
    builders = {
        "vgg16": tf.keras.applications.VGG16,
        "mobilenetv2": tf.keras.applications.MobileNetV2,
        "inceptionv3": tf.keras.applications.InceptionV3,
        "resnet152v2": tf.keras.applications.ResNet152V2,
    }
    builder = builders[MODEL_SLUG]
    last_error = None
    for weights in ("imagenet", None):
        try:
            if weights is None:
                print("Using randomly initialized image base because ImageNet weights were unavailable.")
            else:
                print("Using ImageNet image base fallback.")
            return builder(include_top=False, weights=weights, input_shape=(*IMAGE_SIZE, 3))
        except Exception as exc:
            last_error = exc
            if weights == "imagenet":
                print(f"ImageNet base load failed: {exc}")
    raise RuntimeError("Could not build fallback image base") from last_error


def load_pretrained_checkpoint_and_base() -> tuple[tf.keras.Model | None, tf.keras.Model]:
    if CHECKPOINT_PATH is None:
        print("No local checkpoint found; falling back to an application image base.")
        return None, build_application_image_base()

    try:
        old_model = tf.keras.models.load_model(CHECKPOINT_PATH, compile=False)
        image_base = old_model.get_layer(BASE_LAYER_NAME)
        if not isinstance(image_base, tf.keras.Model):
            raise TypeError(f"Layer {BASE_LAYER_NAME!r} is not a nested Keras Model")
        print(f"Loaded pretrained image base from: {CHECKPOINT_PATH}")
        return old_model, image_base

    except Exception as exc:
        print(f"Could not load checkpoint on this Kaggle runtime: {exc}")
        print("Falling back to a fresh application image base.")
        return None, build_application_image_base()

def build_fusion_model(train_cbc_values: np.ndarray) -> tuple[tf.keras.Model, tf.keras.Model, tf.keras.Model]:
    regularizer = tf.keras.regularizers.l2(1e-4)

    old_model, image_base = load_pretrained_checkpoint_and_base()
    image_base.trainable = False

    image_input = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name="image_input")
    x = BackbonePreprocessing(PREPROCESS_MODE, name=f"{MODEL_SLUG}_preprocess")(image_input)
    x = image_base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name="image_global_average_pooling")(x)
    x = tf.keras.layers.Dense(
        256,
        activation="relu",
        kernel_regularizer=regularizer,
        name="image_embedding_dense",
    )(x)
    image_embedding = tf.keras.layers.Dropout(0.30, name="image_embedding_dropout")(x)

    cbc_input = tf.keras.Input(shape=(len(CBC_FEATURES),), name="cbc_input")
    normalizer = tf.keras.layers.Normalization(axis=-1, name="cbc_normalization")
    normalizer.adapt(train_cbc_values.astype(np.float32))

    y = normalizer(cbc_input)
    y = tf.keras.layers.Dense(64, activation="relu", kernel_regularizer=regularizer, name="cbc_dense_64")(y)
    y = tf.keras.layers.Dropout(0.25, name="cbc_dropout_64")(y)
    y = tf.keras.layers.Dense(32, activation="relu", kernel_regularizer=regularizer, name="cbc_dense_32")(y)
    cbc_embedding = tf.keras.layers.Dropout(0.10, name="cbc_dropout_32")(y)

    fused = tf.keras.layers.Concatenate(name="fusion_concat")([image_embedding, cbc_embedding])
    z = tf.keras.layers.Dense(128, activation="relu", kernel_regularizer=regularizer, name="fusion_dense_128")(fused)
    z = tf.keras.layers.Dropout(0.40, name="fusion_dropout_128")(z)
    z = tf.keras.layers.Dense(64, activation="relu", kernel_regularizer=regularizer, name="fusion_dense_64")(z)
    z = tf.keras.layers.Dropout(0.20, name="fusion_dropout_64")(z)
    output = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32", name="classifier")(z)

    model = tf.keras.Model(
        inputs=[image_input, cbc_input],
        outputs=output,
        name=f"{MODEL_SLUG}_image_cbc_fusion",
    )
    return model, image_base, old_model


def compile_model(model: tf.keras.Model, learning_rate: float) -> None:
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

Assertions And Fine-Tuning Policy
This section enforces the requested checks:

output shape (batch, 4)
old classifier head is not reused
Stage 1 fully freezes CNN base
Stage 2 unfreezes only intended top layers

In [14]:
def assert_model_output_shape(model: tf.keras.Model, dataset: tf.data.Dataset) -> tuple[int, int]:
    batch = next(iter(dataset))
    inputs = batch[0]
    logits = model(inputs, training=False)
    if logits.shape.rank != 2 or int(logits.shape[-1]) != NUM_CLASSES:
        raise AssertionError(f"Expected output shape (batch, {NUM_CLASSES}), got {logits.shape}")
    return int(logits.shape[0]), int(logits.shape[1])


def assert_old_head_not_reused(new_model: tf.keras.Model, old_model: tf.keras.Model | None) -> None:
    if old_model is None:
        print("No checkpoint model loaded; old-head reuse check skipped.")
        return

    old_head_names = [
        layer.name
        for layer in old_model.layers
        if layer.name not in {BASE_LAYER_NAME, old_model.layers[0].name}
    ]
    overlap = sorted(set(old_head_names) & {layer.name for layer in new_model.layers})
    if overlap:
        raise AssertionError(
            "Old classifier/head layers appear to be reused in new model: "
            + ", ".join(overlap)
        )


def _is_weight_bearing(layer: tf.keras.layers.Layer) -> bool:
    return len(layer.weights) > 0


def assert_stage1_frozen(image_base: tf.keras.Model) -> None:
    trainable_layers = [
        layer.name
        for layer in image_base.layers
        if layer.trainable and _is_weight_bearing(layer)
    ]
    if trainable_layers:
        raise AssertionError(
            "Stage 1 expected fully frozen image base, but found trainable layers: "
            + ", ".join(trainable_layers[:10])
        )


def configure_top_layer_fine_tuning(image_base: tf.keras.Model) -> list[str]:
    image_base.trainable = True
    for layer in image_base.layers:
        layer.trainable = False

    if FINE_TUNE_PREFIXES:
        for layer in image_base.layers:
            if layer.name.startswith(FINE_TUNE_PREFIXES):
                layer.trainable = True
    elif FINE_TUNE_LAST_N > 0:
        non_bn_layers = [
            layer
            for layer in image_base.layers
            if not isinstance(layer, tf.keras.layers.BatchNormalization) and _is_weight_bearing(layer)
        ]
        for layer in non_bn_layers[-FINE_TUNE_LAST_N:]:
            layer.trainable = True

    for layer in image_base.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

    return [
        layer.name
        for layer in image_base.layers
        if layer.trainable and _is_weight_bearing(layer)
    ]


def assert_stage2_policy(image_base: tf.keras.Model) -> None:
    trainable_layers = [
        layer
        for layer in image_base.layers
        if layer.trainable and _is_weight_bearing(layer)
    ]
    trainable_names = [layer.name for layer in trainable_layers]

    bn_trainable = [layer.name for layer in trainable_layers if isinstance(layer, tf.keras.layers.BatchNormalization)]
    if bn_trainable:
        raise AssertionError("BatchNorm layers must remain frozen: " + ", ".join(bn_trainable))

    if FINE_TUNE_PREFIXES:
        invalid = [
            name
            for name in trainable_names
            if not name.startswith(FINE_TUNE_PREFIXES)
        ]
        if invalid:
            raise AssertionError(
                "Found trainable layers outside requested prefix policy: " + ", ".join(invalid)
            )
    elif FINE_TUNE_LAST_N > 0:
        non_bn_layers = [
            layer
            for layer in image_base.layers
            if not isinstance(layer, tf.keras.layers.BatchNormalization) and _is_weight_bearing(layer)
        ]
        expected = {layer.name for layer in non_bn_layers[-FINE_TUNE_LAST_N:]}
        actual = set(trainable_names)
        if actual != expected:
            missing = sorted(expected - actual)
            extra = sorted(actual - expected)
            raise AssertionError(
                f"Stage 2 layer policy mismatch. Missing: {missing[:5]}, Extra: {extra[:5]}"
            )

1-Epoch Smoke Training
This runs a quick sanity-check on a small subset before full training.

In [15]:
if RUN_SMOKE_TEST:
    print("Running 1-epoch smoke training...")

    smoke_train_df = train_df.head(SMOKE_TRAIN_ROWS).copy()
    smoke_val_df = val_df.head(SMOKE_VAL_ROWS).copy()

    smoke_weight_map = compute_inverse_frequency_weights(
        smoke_train_df["final_class"].to_numpy(dtype=np.int64)
    )

    smoke_train_ds = make_dataset(
        smoke_train_df,
        batch_size=max(4, min(BATCH_SIZE, 8)),
        shuffle=True,
        seed=SEED,
        sample_weight_map=smoke_weight_map,
    )
    smoke_val_ds = make_dataset(
        smoke_val_df,
        batch_size=max(4, min(BATCH_SIZE, 8)),
        shuffle=False,
        seed=SEED,
    )

    smoke_model, smoke_image_base, smoke_old_model = build_fusion_model(
        train_df[CBC_FEATURES].to_numpy(dtype=np.float32)
    )
    assert_stage1_frozen(smoke_image_base)
    assert_old_head_not_reused(smoke_model, smoke_old_model)
    assert_model_output_shape(smoke_model, smoke_val_ds)

    compile_model(smoke_model, learning_rate=1e-4)
    smoke_model.fit(smoke_train_ds, validation_data=smoke_val_ds, epochs=1, verbose=1)
    print("Smoke training completed.")

    tf.keras.backend.clear_session()
else:
    print("Smoke training is disabled.")

Running 1-epoch smoke training...


I0000 00:00:1778685667.270654      23 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Could not load checkpoint on this Kaggle runtime: <class 'keras.src.models.functional.Functional'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras.src.models.functional', 'class_name': 'Functional', 'config': {}, 'registered_name': 'Functional', 'build_config': {'input_shape': None}, 'compile_config': None}.

Exception encountered: <class 'keras.src.layers.core.dense.Dense'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras.layers', 'class_name': 'Dense', 'config': {'name': 'dense', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name':

I0000 00:00:1778685671.383217      23 cuda_dnn.cc:529] Loaded cuDNN version 91002


8/8 ━━━━━━━━━━━━━━━━━━━━ 11s 286ms/step - accuracy: 0.0958 - loss: 0.6691 - val_accuracy: 0.7188 - val_loss: 1.1839
Smoke training completed.


Full Two-Stage Training
Stage 1 trains with frozen CNN base. Stage 2 unfreezes only configured top layers.

In [16]:
def history_to_frame(history: tf.keras.callbacks.History, stage: str, start_epoch: int) -> pd.DataFrame:
    rows = []
    for idx in range(len(history.epoch)):
        row = {"epoch": start_epoch + idx + 1, "stage": stage}
        for key, values in history.history.items():
            row[key] = values[idx]
        rows.append(row)
    return pd.DataFrame(rows)


def collect_labels(dataset: tf.data.Dataset) -> np.ndarray:
    labels = []
    for batch in dataset:
        labels.append(batch[1].numpy())
    return np.concatenate(labels).astype(np.int64)


def predict_labels(model: tf.keras.Model, dataset: tf.data.Dataset) -> np.ndarray:
    input_ds = dataset.map(lambda inputs, labels: inputs)
    probabilities = model.predict(input_ds, verbose=0)
    return probabilities.argmax(axis=1).astype(np.int64)


train_ds = make_dataset(
    train_df,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    sample_weight_map=class_weight_map,
)
val_ds = make_dataset(
    val_df,
    batch_size=BATCH_SIZE,
    shuffle=False,
    seed=SEED,
)
test_ds = make_dataset(
    test_df,
    batch_size=BATCH_SIZE,
    shuffle=False,
    seed=SEED,
)

model, image_base, old_model = build_fusion_model(train_df[CBC_FEATURES].to_numpy(dtype=np.float32))

assert_stage1_frozen(image_base)
assert_old_head_not_reused(model, old_model)
output_shape = assert_model_output_shape(model, val_ds)
print(f"Output shape check passed: {output_shape}")

checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath=MODEL_SAVE_PATH,
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    verbose=1,
)

history_frames = []

if STAGE1_EPOCHS > 0:
    print("Stage 1: frozen CNN base")
    compile_model(model, learning_rate=1e-4)
    stage1_es = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1,
    )
    stage1_history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=STAGE1_EPOCHS,
        callbacks=[checkpoint_cb, stage1_es],
        verbose=1,
    )
    history_frames.append(history_to_frame(stage1_history, "stage1_frozen_image_encoder", 0))

if STAGE2_EPOCHS > 0:
    print("Stage 2: top-layer fine-tuning")
    trainable_layer_names = configure_top_layer_fine_tuning(image_base)
    assert_stage2_policy(image_base)
    print(f"Trainable CNN layers in stage 2: {len(trainable_layer_names)}")

    compile_model(model, learning_rate=1e-5)
    stage2_es = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1,
    )
    start_epoch = int(sum(len(frame) for frame in history_frames))
    stage2_history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=STAGE2_EPOCHS,
        callbacks=[checkpoint_cb, stage2_es],
        verbose=1,
    )
    history_frames.append(history_to_frame(stage2_history, "stage2_top_layer_finetune", start_epoch))

if not history_frames:
    raise ValueError("Nothing was trained. Increase STAGE1_EPOCHS and/or STAGE2_EPOCHS.")

history_df = pd.concat(history_frames, ignore_index=True)
history_df.to_csv(HISTORY_PATH, index=False)
print(f"Saved training history: {HISTORY_PATH}")

if MODEL_SAVE_PATH.exists():
    best_model = tf.keras.models.load_model(
        MODEL_SAVE_PATH,
        custom_objects={"BackbonePreprocessing": BackbonePreprocessing},
        compile=False,
    )
else:
    model.save(MODEL_SAVE_PATH)
    best_model = model

print(f"Best model path: {MODEL_SAVE_PATH}")

Could not load checkpoint on this Kaggle runtime: <class 'keras.src.models.functional.Functional'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras.src.models.functional', 'class_name': 'Functional', 'config': {}, 'registered_name': 'Functional', 'build_config': {'input_shape': None}, 'compile_config': None}.

Exception encountered: <class 'keras.src.layers.core.dense.Dense'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras.layers', 'class_name': 'Dense', 'config': {'name': 'dense', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name':

Test Evaluation And Artifact Export
Saves:

best .keras model
history CSV
classification report
confusion matrix
final comparison CSV (fusion_results_summary.csv)

In [17]:
y_true = collect_labels(test_ds)
y_pred = predict_labels(best_model, test_ds)

report_text = classification_report(
    y_true,
    y_pred,
    labels=np.arange(NUM_CLASSES),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0,
)
report_dict = classification_report(
    y_true,
    y_pred,
    labels=np.arange(NUM_CLASSES),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)

accuracy = accuracy_score(y_true, y_pred)
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    labels=np.arange(NUM_CLASSES),
    average="macro",
    zero_division=0,
)
_, _, weighted_f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    labels=np.arange(NUM_CLASSES),
    average="weighted",
    zero_division=0,
)

best_val_accuracy = float(history_df["val_accuracy"].max()) if "val_accuracy" in history_df else float("nan")
best_val_loss = float(history_df["val_loss"].min()) if "val_loss" in history_df else float("nan")

cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
fig, ax = plt.subplots(figsize=(8, 7))
cm_display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
cm_display.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
ax.set_title(f"{MODEL_DISPLAY_NAME} Image + CBC Fusion Confusion Matrix ({TRANSFORMED_SUBSET_NAME})")
fig.tight_layout()
fig.savefig(CONFUSION_MATRIX_PATH, dpi=150)
plt.close(fig)

with REPORT_PATH.open("w", encoding="utf-8") as handle:
    handle.write(f"Dataset: {TRANSFORMED_SUBSET_NAME}\n")
    handle.write(f"Backbone: {MODEL_DISPLAY_NAME}\n")
    handle.write(f"Data root: {DATASET_ROOT}\n")
    handle.write(f"Checkpoint: {CHECKPOINT_PATH if CHECKPOINT_PATH else 'application fallback'}\n")
    handle.write(f"Test accuracy: {accuracy:.6f}\n")
    handle.write(f"Macro F1: {macro_f1:.6f}\n")
    handle.write(f"Weighted F1: {weighted_f1:.6f}\n")
    handle.write("\n")
    handle.write(report_text)

summary_row = {
    "dataset": TRANSFORMED_SUBSET_NAME,
    "run_name": RUN_NAME,
    "backbone": MODEL_DISPLAY_NAME,
    "checkpoint_path": str(CHECKPOINT_PATH) if CHECKPOINT_PATH else "application_fallback",
    "test_accuracy": float(accuracy),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "macro_precision": float(macro_precision),
    "macro_recall": float(macro_recall),
    "best_val_accuracy": float(best_val_accuracy),
    "best_val_loss": float(best_val_loss),
}

if SUMMARY_PATH.exists():
    summary_df = pd.read_csv(SUMMARY_PATH)
else:
    summary_df = pd.DataFrame(columns=list(summary_row.keys()))

if not summary_df.empty and {"dataset", "backbone"}.issubset(summary_df.columns):
    is_same_run = (
        (summary_df["dataset"] == TRANSFORMED_SUBSET_NAME)
        & (summary_df["backbone"] == MODEL_DISPLAY_NAME)
    )
    summary_df = summary_df[~is_same_run]
elif not summary_df.empty and "backbone" in summary_df.columns:
    summary_df = summary_df[summary_df["backbone"] != MODEL_DISPLAY_NAME]

summary_df = pd.concat([summary_df, pd.DataFrame([summary_row])], ignore_index=True)
summary_df.to_csv(SUMMARY_PATH, index=False)

print("Evaluation complete.")
print(f"Classification report: {REPORT_PATH}")
print(f"Confusion matrix     : {CONFUSION_MATRIX_PATH}")
print(f"Summary CSV          : {SUMMARY_PATH}")
print(pd.DataFrame([summary_row]))

print("Per-class metrics:")
per_class_df = pd.DataFrame(report_dict).T
print(per_class_df.loc[CLASS_NAMES, ["precision", "recall", "f1-score", "support"]])

Evaluation complete.
Classification report: /kaggle/working/Code/newFusionModel/multiClassImageClassification/artifacts/transformed_aneRBC_ii/metrics/multiClass_transformed_aneRBC_ii_mobilenetv2_classification_report.txt
Confusion matrix     : /kaggle/working/Code/newFusionModel/multiClassImageClassification/artifacts/transformed_aneRBC_ii/plots/multiClass_transformed_aneRBC_ii_mobilenetv2_confusion_matrix.png
Summary CSV          : /kaggle/working/Code/newFusionModel/multiClassImageClassification/artifacts/transformed_aneRBC_ii/fusion_results_summary_transformed_aneRBC_ii.csv
                 dataset                                      run_name  \
0  transformed_AneRBC-II  multiClass_transformed_aneRBC_ii_mobilenetv2   

      backbone                                    checkpoint_path  \
0  MobileNetV2  /kaggle/working/fyp/Code/ImageClassification/a...   

   test_accuracy  macro_f1  weighted_f1  macro_precision  macro_recall  \
0       0.856394   0.75457     0.852242         0.7917

/tmp/ipykernel_23/1790603698.py:88: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  summary_df = pd.concat([summary_df, pd.DataFrame([summary_row])], ignore_index=True)


In [18]:
!ls
!ls -ra

 Code			 requirements-kaggle.txt   transformDataset
 datasetTransformation	 requirements.txt	   viva_questions
 __pycache__		'research papers'
 .vscode	    requirements.txt	      .github		      Code
 viva_questions     requirements-kaggle.txt   .gitattributes	      ..
 transformDataset   __pycache__		      .git		      .
'research papers'   .gitignore		      datasetTransformation
